In [1]:
import pandas as pd
import re
import glob

In [2]:
pen_data = pd.read_csv("unique_pen_combined(2).csv")
kaggle_data = pd.read_csv("merged_dataset.csv")

In [4]:
# Define normalization function
def normalize_title(title):
    if pd.isnull(title):
        return ""
    title = title.lower()
    title = title.replace('&', 'and')  # convert ampersands to 'and'
    title = re.sub(r'\(.*?\)', '', title)  # remove anything in parentheses
    title = re.sub(r'\s+', ' ', title)  # collapse multiple spaces
    title = title.strip()
    return title

In [5]:
pen_data['Normalized Title'] = pen_data['Title'].apply(normalize_title)
kaggle_data['Normalized Title'] = kaggle_data['Title'].apply(normalize_title)

In [6]:
filtered_kaggle_data = kaggle_data[
    ((kaggle_data['Normalized Title'].isin(pen_data['Normalized Title'])) & (kaggle_data['Banned'] == 1)) |
    (~kaggle_data['Normalized Title'].isin(pen_data['Normalized Title']) & (kaggle_data['Banned'] == 0))
]

filtered_kaggle_data['Banned'].value_counts()

Banned
0    8757
1    7516
Name: count, dtype: int64

In [7]:
# get titles from pen_data that is missing
missing_titles = pen_data[~pen_data['Normalized Title'].isin(filtered_kaggle_data['Normalized Title'])]

# take columns we want
missing_titles = missing_titles[['Title', 'Normalized Title', 'Author']]

# mark banned status
missing_titles['Banned'] = 1

# Append the missing titles to the original kaggle_data
updated_kaggle_data = pd.concat([filtered_kaggle_data, missing_titles], ignore_index=True)

updated_kaggle_data['Banned'].value_counts()

Banned
1    10968
0     8757
Name: count, dtype: int64

In [ ]:
# Get all matching JSON files
json_files = glob.glob('/Users/ariannahaider/Downloads/SML-IW-Book-Ban/books_info_finals(2)/books_info_final_*.json')

# Load and combine data from all matching files
data_json = []
for file in json_files:
    with open(file, 'r') as f:
        file_data = json.load(f)
        if isinstance(file_data, list):
            data_json.extend(file_data)

all_titles = []
all_categories = []

if isinstance(data_json, list):
    for item in data_json:
        volume_info = item.get("volumeInfo", {})
        title = volume_info.get("title", [])
        categories = volume_info.get("categories", [])

        unique_categories = []
        seen_categories = set()
        for category in categories:
            sections = category.split(' / ')
            for section in sections:
                if section not in seen_categories:
                    unique_categories.append(section)
                    seen_categories.add(section)

        all_titles.append(title)
        all_categories.append(unique_categories)

cat_df = pd.DataFrame({
    'Title': all_titles,
    'Categories': all_categories
})